In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:
import json
import re
import urllib.request
from datetime import date, datetime

from common_utils.logging import get_logger
from common_utils.ingestor import (
    setup_catalog,
    setup_schema,
    setup_volume,
    setup_directory,
    read_and_write_raw,
    list_gdrive_entries,
)
from common_utils.bronze import read_raw, bronze_ingestor

logger = get_logger("dim-fact-ingestion")

# ============================================================
# 1. READ THE CONFIG
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

with open(config_path) as f:
    config = json.load(f)

catalog          = config["target"]["catalog"]
schema           = config["target"]["schema"]
volume           = config["target"]["volume"]
base_path        = config["target"]["base_path"]

parent_folder_url  = config["gdrive"]["parent_folder_url"]
folder_listing_url = config["gdrive"]["folder_listing_url"]
download_url       = config["gdrive"]["base_download_url"]
dimensions_folder  = config["gdrive"]["dimensions_folder_name"]
date_format        = config["gdrive"]["date_folder_format"]

watermark_table  = config["watermark"]["table"]

file_format      = config["load"]["file_format"]
read_options     = config["load"]["read_options"]
write_mode       = config["load"]["mode"]

run_date = date.today().isoformat()

# ============================================================
# 2. CREATE CATALOG, SCHEMA, VOLUME AND WATERMARK TABLE
# ============================================================
setup_catalog(spark, catalog)
setup_schema(spark, catalog, schema)
setup_volume(spark, catalog, schema, volume)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {watermark_table} (
        source_name  STRING,
        batch_date   STRING,
        file_name    STRING,
        file_id      STRING,
        folder_id    STRING,
        ingested_at  TIMESTAMP
    )
""")

# ============================================================
# 3. READ THE WATERMARK - WHAT HAS ALREADY BEEN INGESTED
# ============================================================
already_ingested = []
for row in spark.sql(f"""
    SELECT source_name FROM {watermark_table}
    WHERE ingested_at >= current_date()
""").collect():
    already_ingested.append(row["source_name"])

logger.info("Ingested today: %s", already_ingested)

# ============================================================
# 4. FIND THE SUBFOLDERS IN THE PARENT DRIVE FOLDER
#    Result looks like: {"Dimensions": "1L0D...", "01-04-2026": "1v43...", ...}
# ============================================================
parent_folder_id = re.search(r"/folders/([-\w]+)", parent_folder_url).group(1)

subfolders = {}
for entry in list_gdrive_entries(urllib, folder_listing_url, parent_folder_id):
    if entry["is_folder"]:
        subfolders[entry["name"]] = entry["id"]

logger.info("Folders found in Drive: %s", list(subfolders.keys()))

# ============================================================
# 5. FUNCTION: INGEST ALL NEW FILES FROM ONE DRIVE FOLDER
# ============================================================
def ingest_folder(folder_id, raw_folder, batch_date):

    for entry in list_gdrive_entries(urllib, folder_listing_url, folder_id):

        file_name   = entry["name"]                  # e.g. FactWork_02_04_2026.csv
        file_id     = entry["id"]
        source_name = file_name.replace(f".{file_format}", "")   # e.g. FactWork_02_04_2026

        # skip anything that is not a csv file
        if entry["is_folder"] or not file_name.endswith(f".{file_format}"):
            continue

        # skip files that were ingested in an earlier run
        if source_name in already_ingested:
            logger.info("%s - already ingested, skipping", source_name)
            continue

        # step 1: download the file from Drive into the Volume
        landing_folder = f"{base_path}/load_date={run_date}/{raw_folder}"
        setup_directory(dbutils, landing_folder)
        read_and_write_raw(urllib, download_url, landing_folder, {file_name: file_id})

        # step 3: record it in the watermark table so it is skipped next time
        spark.sql(f"""
            INSERT INTO {watermark_table} VALUES
            ('{source_name}', '{batch_date}', '{file_name}', '{file_id}', '{folder_id}', current_timestamp())
        """)

        already_ingested.append(source_name)

        logger.info("%s file ingested ", source_name)

# ============================================================
# 6. INGEST THE DIMENSIONS FOLDER
# ============================================================
if dimensions_folder not in subfolders:
    raise ValueError(f"'{dimensions_folder}' folder not found in Drive")

ingest_folder(subfolders[dimensions_folder], dimensions_folder, "static")

# ============================================================
# 7. INGEST EVERY DATE FOLDER, OLDEST FIRST
# ============================================================
date_folders = []
for folder_name in subfolders:
    try:
        folder_date = datetime.strptime(folder_name, date_format)   # e.g. "02-04-2026" -> date
        date_folders.append((folder_date, folder_name))
    except ValueError:
        pass                                                         # not a date folder, ignore it

date_folders.sort()                                                  # oldest date first

for folder_date, folder_name in date_folders:
    raw_folder = "Fact_" + folder_name.replace("-", "_")     
    ingest_folder(subfolders[folder_name], raw_folder, folder_name)

logger.info("Incremental ingestion complete")

In [0]:
"""spark.sql("DROP VOLUME IF EXISTS employeedataplatform.bronze.raw_data")
spark.sql(f"DELETE FROM {watermark_table}")"""

In [0]:
"""%sql
DROP TABLE employeedataplatform.bronze.dimcancellationreason;
DROP TABLE employeedataplatform.bronze.dimcounty;
DROP TABLE employeedataplatform.bronze.dimpeople;
DROP TABLE employeedataplatform.bronze.dimshrinkagetype;"""